In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1. Đọc dữ liệu
print("Đang đọc dữ liệu train.csv...")
train = pd.read_csv("train.csv")

target_col = "is_fraud"
id_col = "id"
feature_cols = [c for c in train.columns if c not in [target_col, id_col]]

X = train[feature_cols]
y = train[target_col].astype(int)

# 2. Tách cột số và cột phân loại
num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = [c for c in feature_cols if c not in num_cols]

# 3. Pipeline tiền xử lý
preprocessor = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]), num_cols),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])

# 4. Cấu hình MLPClassifier
model = Pipeline([
    ("preprocess", preprocessor),
    ("mlp", MLPClassifier(
        hidden_layer_sizes=(128, 64, 32, 16),
        activation="relu",
        solver="adam",
        batch_size=1024,
        learning_rate_init=0.001,
        max_iter=50,
        early_stopping=True,
        n_iter_no_change=10,
        random_state=42,
        verbose=True
    )),
])

# 5. Chia tập Train - Validation (80% train, 20% val với stratify theo nhãn y)
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Kích thước X_train: {X_train.shape}, X_val: {X_val.shape}")

Đang đọc dữ liệu train.csv...
Kích thước X_train: (559968, 24), X_val: (139992, 24)


In [2]:
# 6. Huấn luyện mô hình
print("Bắt đầu huấn luyện MLP...")
model.fit(X_train, y_train)

# 7. Đánh giá trên tập Validation theo 4 metrics
val_prob = model.predict_proba(X_val)[:, 1]
threshold = 0.5
val_pred = (val_prob >= threshold).astype(int)

roc_auc = roc_auc_score(y_val, val_prob)
auc_pr = average_precision_score(y_val, val_prob)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)

print("\n" + "="*40)
print("--- ĐÁNH GIÁ TRÊN TẬP VALIDATION (0.2) ---")
print(f"ROC-AUC   : {roc_auc:.4f}")
print(f"AUC-PR    : {auc_pr:.4f}")
print(f"Precision : {precision:.4f} (với threshold = {threshold})")
print(f"Recall    : {recall:.4f} (với threshold = {threshold})")
print("="*40)

print("\nBáo cáo chi tiết Classification Report:")
print(classification_report(y_val, val_pred, digits=4))

Bắt đầu huấn luyện MLP...
Iteration 1, loss = 0.05453502
Validation score: 0.986696
Iteration 2, loss = 0.04729346
Validation score: 0.986749
Iteration 3, loss = 0.04676576
Validation score: 0.986624
Iteration 4, loss = 0.04641856
Validation score: 0.986767
Iteration 5, loss = 0.04628435
Validation score: 0.986928
Iteration 6, loss = 0.04587831
Validation score: 0.986999
Iteration 7, loss = 0.04555336
Validation score: 0.986731
Iteration 8, loss = 0.04520810
Validation score: 0.986821
Iteration 9, loss = 0.04475264
Validation score: 0.986856
Iteration 10, loss = 0.04443608
Validation score: 0.986821
Iteration 11, loss = 0.04392708
Validation score: 0.986624
Iteration 12, loss = 0.04348947
Validation score: 0.986660
Iteration 13, loss = 0.04288010
Validation score: 0.986571
Iteration 14, loss = 0.04235558
Validation score: 0.986553
Iteration 15, loss = 0.04163223
Validation score: 0.986499
Iteration 16, loss = 0.04108798
Validation score: 0.986446
Validation score did not improve more t

In [4]:
# 8. Dự đoán trên tập test.csv và xuất file nộp
try:
    test = pd.read_csv("test.csv")
    test_prob = model.predict_proba(test[feature_cols])[:, 1]

    submission = pd.DataFrame({
        id_col: test[id_col],
        target_col: test_prob,
    })
    submission.to_csv("submission_mlp.csv", index=False)
    print("Đã lưu kết quả dự đoán vào submission_mlp.csv thành công!")
except Exception as e:
    print("Bỏ qua bước dự đoán test:", e)

Đã lưu kết quả dự đoán vào submission_mlp.csv thành công!
